## SecciÃ³n 1: ConfiguraciÃ³n del Entorno

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import gc
import json
import warnings
import cv2
from tqdm import tqdm
import zipfile
import shutil

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve, 
                           matthews_corrcoef, balanced_accuracy_score,
                           precision_recall_curve, average_precision_score,
                           f1_score, recall_score, precision_score)
from sklearn.utils import class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping, 
                                       ReduceLROnPlateau, TensorBoard, Callback)
import tensorflow.keras.backend as K

warnings.filterwarnings('ignore')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU configurada: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("No se detectÃ³ ninguna GPU, usando CPU")

AUTO = tf.data.experimental.AUTOTUNE

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

2026-05-10 19:23:20.034073: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778441000.057291    8311 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778441000.065103    8311 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778441000.084064    8311 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778441000.084087    8311 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778441000.084089    8311 computation_placer.cc:177] computation placer alr

GPU configurada: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## SecciÃ³n 2: Funciones Base (El Motor de Datos)

In [2]:
def decode_image(image_data):
    # Decodifica el JPEG empaquetado
    image = tf.image.decode_jpeg(image_data, channels=3)
    # Convierte a float32 y normaliza de 0-255 a 0-1
    image = tf.cast(image, tf.float32) / 255.0
    # Asegura que TensorFlow conozca el tamaÃ±o exacto usando la variable global IMAGE_SIZE
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

def read_labeled_tfrecord(example):
    LABELED_TFREC_FORMAT = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "target": tf.io.FixedLenFeature([], tf.int64),
        "age_approx": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
        "sex": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
        "anatom_site_general_challenge": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
    }
    example = tf.io.parse_single_example(example, LABELED_TFREC_FORMAT)
    
    image = decode_image(example['image'])
    
    # Preprocesamiento de metadatos (Normalización básica)
    age = tf.cast(example['age_approx'], tf.float32) / 100.0  
    sex = tf.cast(example['sex'], tf.float32)                 
    site = tf.cast(example['anatom_site_general_challenge'], tf.float32) / 7.0 
    
    metadata = tf.stack([age, sex, site])
    target = tf.cast(example['target'], tf.float32)
    
    return (image, metadata), target

def data_augment(inputs, target):
    # 1. Desempaquetamos las entradas
    image, metadata = inputs
    
    # 2. Hacemos la aumentación SOLO a la imagen
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_hue(image, max_delta=0.01)
    image = tf.image.random_saturation(image, lower=0.7, upper=1.3)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_brightness(image, max_delta=0.1)
    
    # 3. Volvemos a empaquetar todo exactamente como el modelo lo espera
    return (image, metadata), target

## SecciÃ³n 3: Fase 1 - Entrenamiento Base (256x256)

In [3]:
IMAGE_SIZE = [256, 256]
BATCH_SIZE = 64

# Rutas a los datasets en Kaggle (asegÃºrate de haber aÃ±adido ambos)
GCS_PATH_2020 = '/kaggle/input/datasets/cdeotte/melanoma-256x256'
GCS_PATH_2019 = '/kaggle/input/datasets/cdeotte/isic2019-256x256'

# Obtenemos las listas de todos los archivos .tfrec
FILES_2020 = tf.io.gfile.glob(GCS_PATH_2020 + '/train*.tfrec')
FILES_2019 = tf.io.gfile.glob(GCS_PATH_2019 + '/train*.tfrec')

# Ordenarlos para asegurar reproducibilidad
FILES_2020.sort()
FILES_2019.sort()

# Separar el 20% de 2020 para validaciÃ³n
split_index = int(len(FILES_2020) * 0.8)
VALIDATION_FILENAMES = FILES_2020[split_index:]

# El entrenamiento son los restantes de 2020 + TODOS los de 2019
TRAINING_FILENAMES = FILES_2020[:split_index] + FILES_2019

print(f"Archivos TFRecord para Entrenamiento: {len(TRAINING_FILENAMES)}")
print(f"Archivos TFRecord para ValidaciÃ³n: {len(VALIDATION_FILENAMES)}")

def get_training_dataset():
    dataset = tf.data.TFRecordDataset(TRAINING_FILENAMES, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.map(data_augment, num_parallel_calls=AUTO)
    dataset = dataset.repeat() # Infinito para las Ã©pocas
    dataset = dataset.shuffle(2048) # Mezclador en memoria
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO) # Precarga
    return dataset

def get_validation_dataset():
    dataset = tf.data.TFRecordDataset(VALIDATION_FILENAMES, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.cache() # Guarda en RAM para validar rapidÃ­simo
    dataset = dataset.prefetch(AUTO)
    return dataset

# --- INSTANCIAMOS LOS DATASETS ---
train_generator = get_training_dataset()
val_generator = get_validation_dataset()

# Calculamos los pasos por Ã©poca
NUM_TRAINING_IMAGES = len(TRAINING_FILENAMES) * 2071
NUM_VALIDATION_IMAGES = len(VALIDATION_FILENAMES) * 2071
STEPS_PER_EPOCH = NUM_TRAINING_IMAGES // BATCH_SIZE

print(f"ImÃ¡genes de entrenamiento aproximadas: {NUM_TRAINING_IMAGES:,}")
print(f"ImÃ¡genes de validaciÃ³n aproximadas: {NUM_VALIDATION_IMAGES:,}")
print(f"Pasos por Ã©poca calculados: {STEPS_PER_EPOCH}")

Archivos TFRecord para Entrenamiento: 42
Archivos TFRecord para ValidaciÃ³n: 3


I0000 00:00:1778441010.298024    8311 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


ImÃ¡genes de entrenamiento aproximadas: 86,982
ImÃ¡genes de validaciÃ³n aproximadas: 6,213
Pasos por Ã©poca calculados: 1359


## SecciÃ³n 4: Focal Loss y MÃ©tricas

In [4]:
def focal_loss(gamma=2.0, alpha=0.75):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        
        y_true = tf.cast(y_true, tf.float32)
        
        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_factor = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        
        cross_entropy = -tf.math.log(p_t)
        weight = alpha_factor * tf.pow((1 - p_t), gamma)
        
        loss = weight * cross_entropy
        return tf.reduce_mean(loss)
    return focal_loss_fixed

class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()
        self.recall = tf.keras.metrics.Recall()
        
    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)
        
    def result(self):
        precision = self.precision.result()
        recall = self.recall.result()
        return 2 * ((precision * recall) / (precision + recall + tf.keras.backend.epsilon()))
    
    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

## SecciÃ³n 5: Arquitectura (EfficientNetV2)

In [5]:
def build_multimodal_model(img_size=256, num_metadata=3):
    # --- Rama 1: Procesamiento de Imagen ---
    input_img = layers.Input(shape=(img_size, img_size, 3), name='input_img')
    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    x_img = base_model(input_img)
    x_img = layers.GlobalAveragePooling2D()(x_img)
    
    # --- Rama 2: Procesamiento de Metadatos ---
    input_meta = layers.Input(shape=(num_metadata,), name='input_meta')
    x_meta = layers.Dense(16, activation='relu')(input_meta)
    x_meta = layers.BatchNormalization()(x_meta)
    
    # --- Fusión de ambas ramas ---
    concat = layers.Concatenate()([x_img, x_meta])
    
    # Capas densas finales para la decisión
    x = layers.Dense(128, activation='relu')(concat)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='sigmoid', name='output')(x)
    
    model = models.Model(inputs=[input_img, input_meta], outputs=output)
    return model, base_model

model, base_model = build_multimodal_model(img_size=256)

print(f"ParÃ¡metros totales: {model.count_params():,}")
print(f"ParÃ¡metros entrenables: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

ParÃ¡metros totales: 6,085,585
ParÃ¡metros entrenables: 6,024,945


## SecciÃ³n 6: Entrenamiento Fase 1 (256x256)

In [6]:
EPOCHS_FASE_1 = 5

initial_learning_rate = 0.001
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate,
    decay_steps=STEPS_PER_EPOCH * EPOCHS_FASE_1,
    alpha=0.01
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)

model.compile(
    optimizer=optimizer,
    loss=focal_loss(gamma=2.0, alpha=0.75), 
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print("Modelo compilado y listo para TFRecords")

os.makedirs('checkpoints', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        'checkpoints/best_model_efficientnet.weights.h5',
        monitor='val_pr_auc', 
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_pr_auc',
        patience=3,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]

print("\n" + "="*60)
print("FASE 1: ENTRENAMIENTO INICIAL (TFRECORDS)")
print("="*60)

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS_FASE_1,
    callbacks=callbacks,
    verbose=1
)

Modelo compilado y listo para TFRecords

FASE 1: ENTRENAMIENTO INICIAL (TFRECORDS)
Epoch 1/5


I0000 00:00:1778441061.885309    8382 service.cc:152] XLA service 0x7a7998002f60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778441061.885350    8382 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778441073.180755    8382 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-10 19:24:50.303387: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-10 19:24:50.500439: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-10 19:24:51.053522: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

1359/1359 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.9281 - auc: 0.7995 - loss: 0.0241 - pr_auc: 0.2782 - precision: 0.2939 - recall: 0.3020
Epoch 1: val_pr_auc improved from -inf to 0.04131, saving model to checkpoints/best_model_efficientnet.weights.h5
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 396s 210ms/step - accuracy: 0.9281 - auc: 0.7996 - loss: 0.0241 - pr_auc: 0.2783 - precision: 0.2939 - recall: 0.3021 - val_accuracy: 0.9822 - val_auc: 0.5986 - val_loss: 0.0146 - val_pr_auc: 0.0413 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/5
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.8717 - auc: 0.8667 - loss: 0.0322 - pr_auc: 0.5286 - precision: 0.4695 - recall: 0.5721
Epoch 2: val_pr_auc did not improve from 0.04131
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 261s 192ms/step - accuracy: 0.8717 - auc: 0.8667 - loss: 0.0322 - pr_auc: 0.5286 - precision: 0.4695 - recall: 0.5721 - val_accuracy: 0.8135 - val_auc: 0.6458 - val_loss: 0.0287 - val_pr_auc: 0.0397 - val_precision: 0.0

## SecciÃ³n 7: Entrenamiento Fase 2 - Fine-Tuning (256x256)

In [8]:
print("\n" + "="*60)
print("FASE 2: FINE-TUNING (Descongelando capas)")
print("="*60)

# Descongelamos el modelo base
base_model.trainable = True

# Congelamos las primeras capas para no destruir lo que EfficientNet ya sabe de formas bÃ¡sicas
fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompilamos con un Learning Rate MUCHÃSIMO mÃ¡s bajo (1e-5 en lugar de 1e-4)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print(f"Capas entrenables ahora: {len([l for l in model.layers if l.trainable])}")

# Entrenamos otras 5 Ã©pocas
EPOCHS_FASE_2 = 7

callbacks_fase2 = [
    # Guardamos en el nuevo formato seguro .keras
    ModelCheckpoint(
        'checkpoints/best_model_multimodal.keras', 
        monitor='val_pr_auc', 
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Nuestro paracaídas de seguridad
    EarlyStopping(
        monitor='val_pr_auc',
        patience=4, # Le damos un margen de 4 épocas de paciencia
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]

history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS_FASE_1 + EPOCHS_FASE_2,
    initial_epoch=EPOCHS_FASE_1,
    callbacks=callbacks_fase2,
    verbose=1
)


FASE 2: FINE-TUNING (Descongelando capas)
Capas entrenables ahora: 10
Epoch 6/12
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9475 - auc: 0.9234 - loss: 0.0161 - pr_auc: 0.5261 - precision: 0.4885 - recall: 0.5271
Epoch 6: val_pr_auc improved from -inf to 0.12070, saving model to checkpoints/best_model_multimodal.keras
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 190s 107ms/step - accuracy: 0.9475 - auc: 0.9234 - loss: 0.0161 - pr_auc: 0.5262 - precision: 0.4885 - recall: 0.5271 - val_accuracy: 0.9541 - val_auc: 0.8249 - val_loss: 0.0145 - val_pr_auc: 0.1207 - val_precision: 0.1489 - val_recall: 0.3333
Epoch 7/12
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.8972 - auc: 0.9238 - loss: 0.0250 - pr_auc: 0.6859 - precision: 0.5468 - recall: 0.6995
Epoch 7: val_pr_auc improved from 0.12070 to 0.13184, saving model to checkpoints/best_model_multimodal.keras
1359/1359 ━━━━━━━━━━━━━━━━━━━━ 130s 96ms/step - accuracy: 0.8972 - auc: 0.9238 - loss: 0.0250 - pr_auc: 0.6859 - precision:

## SecciÃ³n 8: EvaluaciÃ³n Intermedia (256x256)

In [9]:
from sklearn.metrics import roc_curve
import numpy as np

print("Extrayendo predicciones del set de ValidaciÃ³n...")

y_true = []
y_pred = []

# Iteramos sobre el dataset de validaciÃ³n que estÃ¡ cacheado en RAM
for (batch_images, batch_meta), batch_labels in val_generator:
    # Le pasamos la lista de ambas entradas al modelo
    preds = model.predict([batch_images, batch_meta], verbose=0)
    y_pred.extend(preds.flatten())
    y_true.extend(batch_labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Encontramos el umbral Ã³ptimo
def find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    valid_idx = np.where(tpr >= min_sensitivity)[0]
    
    if len(valid_idx) > 0:
        optimal_idx = valid_idx[0]
    else:
        optimal_idx = np.argmax(tpr - fpr)
        
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold, fpr[optimal_idx], tpr[optimal_idx]

optimal_threshold, fpr_opt, tpr_opt = find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85)

print(f"\nUmbral Ã³ptimo calculado: {optimal_threshold:.4f}")
print(f"Sensibilidad garantizada: {tpr_opt:.2%}")
print(f"Especificidad lograda: {1-fpr_opt:.2%}")

def evaluate_model_comprehensive(y_true, y_pred, threshold=0.5, suffix=""):
    y_pred_binary = (y_pred >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred_binary)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        'threshold': threshold,
        'auc_roc': roc_auc_score(y_true, y_pred),
        'average_precision': average_precision_score(y_true, y_pred),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'f1_score': f1_score(y_true, y_pred_binary),
        'mcc': matthews_corrcoef(y_true, y_pred_binary),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred_binary),
        'nnd': 1 / (tp / (tp + fn)) if tp > 0 else float('inf'),
        'confusion_matrix': cm
    }
    
    # Imprimir resultados
    print("\n" + "="*60)
    print("EVALUACIÃ“N DEL MODELO")
    print("="*60)
    print(f"Umbral: {threshold:.4f}")
    print(f"\nMÃ©tricas de Rendimiento:")
    print(f"  - AUC-ROC: {metrics['auc_roc']:.4f}")
    print(f"  - Average Precision: {metrics['average_precision']:.4f}")
    print(f"  - Sensibilidad (Recall): {metrics['sensitivity']:.2%}")
    print(f"  - Especificidad: {metrics['specificity']:.2%}")
    print(f"  - PrecisiÃ³n: {metrics['precision']:.2%}")
    print(f"  - NPV: {metrics['npv']:.2%}")
    print(f"  - F1-Score: {metrics['f1_score']:.4f}")
    print(f"  - MCC: {metrics['mcc']:.4f}")
    print(f"  - Balanced Accuracy: {metrics['balanced_accuracy']:.2%}")
    print(f"  - NND: {metrics['nnd']:.2f}")
    return metrics

test_metrics = evaluate_model_comprehensive(y_true, y_pred, optimal_threshold)

Extrayendo predicciones del set de ValidaciÃ³n...

Umbral Ã³ptimo calculado: 0.2002
Sensibilidad garantizada: 85.47%
Especificidad lograda: 69.94%

EVALUACIÃ“N DEL MODELO
Umbral: 0.2002

MÃ©tricas de Rendimiento:
  - AUC-ROC: 0.8541
  - Average Precision: 0.1368
  - Sensibilidad (Recall): 85.47%
  - Especificidad: 69.94%
  - PrecisiÃ³n: 4.91%
  - NPV: 99.62%
  - F1-Score: 0.0929
  - MCC: 0.1585
  - Balanced Accuracy: 77.71%
  - NND: 1.17


## SecciÃ³n 9: Fase 3 - Progressive Resizing a Alta ResoluciÃ³n (384x384)

In [ ]:
print("Limpiando memoria para Progressive Resizing...")
del train_generator
del val_generator
gc.collect()
K.clear_session()

IMAGE_SIZE = [384, 384]
BATCH_SIZE = 16

GCS_PATH_2020_384 = '/kaggle/input/datasets/cdeotte/melanoma-384x384'
GCS_PATH_2019_384 = '/kaggle/input/datasets/cdeotte/isic2019-384x384'

FILES_2020_384 = tf.io.gfile.glob(GCS_PATH_2020_384 + '/train*.tfrec')
FILES_2019_384 = tf.io.gfile.glob(GCS_PATH_2019_384 + '/train*.tfrec')

FILES_2020_384.sort()
FILES_2019_384.sort()

split_index_384 = int(len(FILES_2020_384) * 0.8)
VALIDATION_FILENAMES_384 = FILES_2020_384[split_index_384:]
TRAINING_FILENAMES_384 = FILES_2020_384[:split_index_384] + FILES_2019_384

def get_training_dataset_384():
    dataset = tf.data.TFRecordDataset(TRAINING_FILENAMES_384, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.map(data_augment, num_parallel_calls=AUTO)
    dataset = dataset.repeat()
    dataset = dataset.shuffle(512)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

def get_validation_dataset_384():
    dataset = tf.data.TFRecordDataset(VALIDATION_FILENAMES_384, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

train_generator_384 = get_training_dataset_384()
val_generator_384 = get_validation_dataset_384()

NUM_TRAINING_IMAGES_384 = len(TRAINING_FILENAMES_384) * 2071
NUM_VALIDATION_IMAGES_384 = len(VALIDATION_FILENAMES_384) * 2071
STEPS_PER_EPOCH_384 = NUM_TRAINING_IMAGES_384 // BATCH_SIZE

print(f"Pasos por Ã©poca (384x384): {STEPS_PER_EPOCH_384}")

## SecciÃ³n 10: InyecciÃ³n de Conocimiento y Entrenamiento Final

In [ ]:
print("\n" + "="*60)
print("FASE 3: PROGRESSIVE RESIZING (384x384)")
print("="*60)

model_384, base_model_384 = build_multimodal_model(img_size=384)
model_384.load_weights('checkpoints/best_model_efficientnet.weights.h5')

base_model_384.trainable = True
fine_tune_at_384 = len(base_model_384.layers) - 50
for layer in base_model_384.layers[:fine_tune_at_384]:
    layer.trainable = False

model_384.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6, clipnorm=1.0),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

EPOCHS_FASE_3 = 2

callbacks_phase3 = [
    ModelCheckpoint(
        'checkpoints/best_model_efficientnet_384.weights.h5',
        monitor='val_pr_auc', 
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_phase3 = model_384.fit(
    train_generator_384,
    validation_data=val_generator_384,
    steps_per_epoch=STEPS_PER_EPOCH_384,
    epochs=EPOCHS_FASE_3,
    callbacks=callbacks_phase3,
    verbose=1
)

## SecciÃ³n 11: EvaluaciÃ³n Final y Conclusiones

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

print("Extrayendo predicciones del set de Validación (384x384 Multimodal)...")

# 1. REINICIAMOS LAS LISTAS (Esto soluciona tu error)
y_true = []
y_pred = []

# 2. ITERAMOS SOBRE EL NUEVO GENERADOR DE 384
# Como es multimodal, desempaquetamos (imágenes, metadatos)
for (batch_images, batch_meta), batch_labels in val_generator_384:
    
    # Pasamos AMBAS entradas al modelo
    preds = model_384.predict([batch_images, batch_meta], verbose=0)
    
    # Ahora sí podemos usar extend porque y_pred e y_true vuelven a ser listas
    y_pred.extend(preds.flatten())
    y_true.extend(batch_labels.numpy())

# 3. CONVERTIMOS A NUMPY AL FINAL
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Encontramos el umbral óptimo
def find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    valid_idx = np.where(tpr >= min_sensitivity)[0]
    
    if len(valid_idx) > 0:
        optimal_idx = valid_idx[0]
    else:
        optimal_idx = np.argmax(tpr - fpr)
        
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold, fpr[optimal_idx], tpr[optimal_idx]

optimal_threshold_384, fpr_opt, tpr_opt = find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85)

print(f"\nUmbral óptimo calculado final: {optimal_threshold_384:.4f}")

# 4. LLAMADA A TU FUNCIÓN DE EVALUACIÓN COMPREHENSIVA
# (Asegúrate de pasarle los arrays calculados)
test_metrics = evaluate_model_comprehensive(y_true, y_pred, optimal_threshold_384)

## SecciÃ³n 12: ExportaciÃ³n a TFLite (Opcional)

In [10]:
import tensorflow as tf
import os

print("\n" + "="*60)
print("SECCIÓN 12: EXPORTACIÓN A TFLITE (MODELO MULTIMODAL 256x256)")
print("="*60)

model_name_keras = 'skincare_model_multimodal.keras'
model.save(model_name_keras)
print(f"✓ Modelo guardado en formato Keras: {model_name_keras}")

# 1. Cargamos el modelo ganador de la Fase 2
print("Cargando el mejor modelo de la Fase 2...")
best_model_path = 'checkpoints/best_model_multimodal.keras' # O .keras si lo cambiaste
final_model = tf.keras.models.load_model(best_model_path, compile=False)

# 2. Convertimos el modelo a formato TensorFlow Lite
print("Iniciando conversión a TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)

# Opcional pero MUY RECOMENDADO para móviles: Optimización de peso
# Esto comprime el modelo de ~25MB a unos ~7MB sin perder apenas precisión
converter.optimizations = [tf.lite.Optimize.DEFAULT]

try:
    tflite_model = converter.convert()
    
    # 3. Guardamos el archivo
    tflite_filename = 'skincare_multimodal_256.tflite'
    with open(tflite_filename, 'wb') as f:
        f.write(tflite_model)
        
    # Calculamos el peso en MB
    model_size_mb = os.path.getsize(tflite_filename) / (1024 * 1024)
    print(f"\n¡ÉXITO! Modelo exportado a: {tflite_filename}")
    print(f"Peso final para la app móvil: {model_size_mb:.2f} MB")

except Exception as e:
    print(f"Error durante la conversión: {e}")


SECCIÓN 12: EXPORTACIÓN A TFLITE (MODELO MULTIMODAL 256x256)
✓ Modelo guardado en formato Keras: skincare_model_multimodal.keras
Cargando el mejor modelo de la Fase 2...
Iniciando conversión a TFLite...
INFO:tensorflow:Assets written to: /tmp/tmpgwtmy0_7/assets


INFO:tensorflow:Assets written to: /tmp/tmpgwtmy0_7/assets


Saved artifact at '/tmp/tmpgwtmy0_7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='input_img'), TensorSpec(shape=(None, 3), dtype=tf.float32, name='input_meta')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134659077878224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134659077879760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134661051084304: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  134661051077008: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  134661051076816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134661051078736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134661051080080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134661051077584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134661051077200: TensorSpec(shape=(), dtype=tf.resource, name=None)

W0000 00:00:1778444445.422176    8311 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778444445.422227    8311 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1778444445.674339    8311 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled



¡ÉXITO! Modelo exportado a: skincare_multimodal_256.tflite
Peso final para la app móvil: 6.45 MB
